# 🧪 Chapter 3: Statistical Experiments and Significance Testing
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 3.1 Introduction to A/B Testing
The core of statistical experimentation is the **A/B Test**. It is used to test a hypothesis about a single parameter (e.g., "Does a blue button generate more clicks than a red button?").

- **Control Group:** The standard, existing version (Page A).
- **Treatment Group:** The new version with the proposed change (Page B).
- **Test Statistic:** The metric we are measuring (e.g., Conversion Rate, Average Order Value).

In traditional statistics, experiments require strict mathematical assumptions. In modern data science, we rely heavily on **Resampling (Permutation Tests)** to let the data speak for itself.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from statsmodels.stats.power import TTestIndPower

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 3.")

## 3.2 Hypothesis Testing and Permutation Tests

When Page B outperforms Page A, we must ask: *"Is this difference real, or did Page B just get lucky due to random chance?"*

- **Null Hypothesis ($H_0$):** The assumption that there is NO real difference between the groups. Any observed difference is purely due to random chance.
- **Alternative Hypothesis ($H_1$):** The assumption that the treatment (Page B) actually had an effect.

### The Permutation Test
To test the Null Hypothesis computationally, we use a Permutation Test:
1. Combine the results from both Group A and Group B into a single pool.
2. Shuffle (permeate) the pool and randomly divide it into two new groups of the same original sizes.
3. Calculate the difference in metrics between these two fake groups.
4. Repeat this thousands of times to build a distribution of differences that could happen *purely by chance*.
5. Compare our actual observed difference to this random distribution.

In [ ]:
# 1. Create Synthetic Data for an E-commerce A/B Test (Session Times in Seconds)
# Page A has a mean of 120s, Page B has a mean of 135s
page_A = np.random.normal(120, 40, size=500)
page_B = np.random.normal(135, 40, size=500)

observed_diff = np.mean(page_B) - np.mean(page_A)
print(f"Observed Difference in Means (B - A): {observed_diff:.2f} seconds\n")

# 2. Write a Custom Permutation Test Function
def permutation_test(group_A, group_B, n_permutations=2000):
    pooled_data = np.concatenate([group_A, group_B])
    size_A = len(group_A)
    
    permuted_diffs = []
    for _ in range(n_permutations):
        np.random.shuffle(pooled_data)
        fake_A = pooled_data[:size_A]
        fake_B = pooled_data[size_A:]
        permuted_diffs.append(np.mean(fake_B) - np.mean(fake_A))
        
    return np.array(permuted_diffs)

# 3. Run the Permutation Test
perm_diffs = permutation_test(page_A, page_B)

# 4. Visualize the Permutation Distribution
plt.figure(figsize=(8, 5))
sns.histplot(perm_diffs, bins=40, color='gray', alpha=0.6)
plt.axvline(observed_diff, color='red', linestyle='--', linewidth=2, label=f'Observed Diff ({observed_diff:.1f}s)')
plt.title('Permutation Test: Distribution of Random Differences (The Null Hypothesis)')
plt.xlabel('Difference in Means (Seconds)')
plt.legend()
plt.show()

## 3.3 Statistical Significance and P-Values

Looking at the graph above, the red line (our actual result) is far to the right, almost completely outside the gray histogram (random chance). To quantify this, we calculate a **p-value**.

- **p-value:** The probability of obtaining a result as extreme as the observed one, *assuming the Null Hypothesis is true*.
- **Alpha ($\alpha$):** The threshold for significance (usually 0.05 or 5%). If $p < \alpha$, we reject the Null Hypothesis.

*Note on Data Science vs. Traditional Stats:* 
In Python, instead of writing permutation tests from scratch every time, we often use the classical **t-test** (`scipy.stats.ttest_ind`), which provides an incredibly close approximation of the exact permutation p-value without the computational heavy lifting.

In [ ]:
# 1. Calculate the Exact P-value from our Permutation Test
# How many random permutations were greater than or equal to our observed difference?
p_value_permutation = np.mean(perm_diffs >= observed_diff)
print(f"Permutation P-Value : {p_value_permutation:.5f}")

# 2. Calculate the P-value using the Classical Two-Sample T-Test
# We use equal_var=False to perform Welch's t-test (more robust)
t_stat, p_value_ttest = stats.ttest_ind(page_B, page_A, equal_var=False, alternative='greater')
print(f"Classical T-Test P-Value: {p_value_ttest:.5f}")

print("\nConclusion: Both methods yield a p-value near 0.0000. Because p < 0.05, we reject the Null Hypothesis.")
print("We confidently conclude that Page B causes a statistically significant increase in session time.")

## 3.4 Multiple Testing (The Alpha Inflation Problem)

What if we test 20 different colors for our button? 
If we use an alpha threshold of 5% (0.05), we expect a 5% chance of a False Positive (Type I Error) on *every single test*.

If we run 20 tests, the probability of getting at least one False Positive is:
$1 - (1 - 0.05)^{20} \approx 64\%$

This is why "data snooping" is dangerous. To fix this, statisticians use the **Bonferroni Adjustment**: simply divide your alpha threshold by the number of tests (e.g., $0.05 / 20 = 0.0025$).

## 3.5 ANOVA (Analysis of Variance)

Instead of running dozens of A/B tests to compare 4 different web pages, which inflates our error rate, we should use **ANOVA**. 

ANOVA tests a single, sweeping Null Hypothesis: *"All groups have the exact same mean."*

It uses the **F-Statistic**, which is the ratio of the variance *between* the groups to the variance *within* the groups. If the groups are truly different, the variance between them will be much larger than the noise inside them.

In [ ]:
# Create data for 4 different web page designs (A, B, C, D)
np.random.seed(10)
design_A = np.random.normal(100, 20, 200)
design_B = np.random.normal(102, 20, 200)
design_C = np.random.normal(98, 20, 200)
design_D = np.random.normal(115, 20, 200) # Design D is clearly superior

df_anova = pd.DataFrame({
    'Time': np.concatenate([design_A, design_B, design_C, design_D]),
    'Design': ['A']*200 + ['B']*200 + ['C']*200 + ['D']*200
})

# 1. Run Classical One-Way ANOVA using Scipy
f_stat, p_value_anova = stats.f_oneway(design_A, design_B, design_C, design_D)

print(f"ANOVA F-Statistic : {f_stat:.2f}")
print(f"ANOVA P-Value     : {p_value_anova:.5e}")
print("Because p < 0.05, we reject the Null Hypothesis. At least one design is significantly different.")

# 2. Visualize with Boxplots
plt.figure(figsize=(8, 5))
sns.boxplot(x='Design', y='Time', data=df_anova, palette='Set2')
plt.title('ANOVA: Comparing Session Times Across 4 Designs')
plt.show()

## 3.6 Chi-Square Test (For Categorical Data)

ANOVA and T-Tests evaluate numeric continuous data (like session times). But what if our metric is a count or a category? (e.g., Clicks vs. No Clicks).

We use **Pearson's Chi-Square Test**. It evaluates whether the observed counts in a contingency table differ significantly from the expected counts (what we would expect if everything was equal and random).

In [ ]:
# Example: We tested 3 different Headlines and recorded the Clicks and No-Clicks
# Contingency Table formulation:
#                Headline 1 | Headline 2 | Headline 3
# Clicks       |    14     |     8      |     25
# No Clicks    |   986     |   992      |    975

contingency_table = np.array([
    [14, 8, 25],     # Clicks
    [986, 992, 975]  # No Clicks
])

# Run Chi-Square Test
chi2_stat, p_val_chi2, dof, expected_table = stats.chi2_contingency(contingency_table)

print("Observed Table:\n", contingency_table)
print("\nExpected Table (If Null Hypothesis was true):\n", np.round(expected_table, 1))
print(f"\nChi-Square Statistic: {chi2_stat:.2f}")
print(f"P-Value: {p_val_chi2:.4f}")

print("\nConclusion: Because p < 0.05, we conclude that the headlines do NOT have equal click-through rates. Headline 3 significantly outperformed the others.")

## 3.7 Power and Sample Size

A common question before running an A/B test is: *"How many users do I need in my test?"*

This is solved using **Power Analysis**, which balances four moving parts:
1. **Sample Size:** The number of observations per group.
2. **Effect Size:** The minimum difference you want to be able to detect (e.g., a 2% lift in conversions).
3. **Significance Level ($\\alpha$):** The false positive rate (usually 0.05).
4. **Power ($1 - \\beta$):** The probability of detecting a real effect if it exists (usually 0.80). A power of 0.80 means a 20% chance of a False Negative (Type II Error).

If you know 3 of these, you can calculate the 4th.

In [ ]:
# Scenario: We want to detect a small Effect Size (Cohen's d = 0.2)
# We want standard Alpha (0.05) and standard Power (0.80)

effect_size = 0.2
alpha = 0.05
power = 0.80

# Initialize the Power analysis object from statsmodels
power_analysis = TTestIndPower()

# Calculate Sample Size
# Note: 'nobs1' stands for number of observations in group 1. 
# We pass 'None' to the parameter we want to solve for.
required_n = power_analysis.solve_power(effect_size=effect_size, 
                                        alpha=alpha, 
                                        power=power, 
                                        ratio=1.0, # 1:1 ratio between Group A and B
                                        alternative='two-sided')

print(f"Required Sample Size PER GROUP: {int(np.ceil(required_n))} users.")
print(f"Total experiment size needed: {int(np.ceil(required_n)) * 2} users.")

print("\nThis tells the Data Scientist exactly how long they need to run the A/B test before concluding it.")

### Conclusion of Chapter 3

Chapter 3 transitions from simply describing data to rigorously testing it. 

Key Takeaways:
1. **Permutation Tests:** An intuitive, computational way to determine if an observed effect is just random noise.
2. **P-Values:** The probability of getting your result purely by chance. Do not treat p < 0.05 as an absolute truth; it is simply a useful threshold.
3. **Multiple Testing:** Running too many tests guarantees false positives unless you adjust your Alpha (Bonferroni).
4. **ANOVA & Chi-Square:** Extensions of the A/B test to multiple groups and categorical metrics.
5. **Power Analysis:** Essential for determining if your experiment actually collected enough data to prove anything.